### Categorize SASB Metrics (quantitative)

In [56]:
import pandas as pd
import numpy as np
import re

In [57]:
df1 = pd.read_excel(r"D:\Dropbox\Dropbox\Dropbox\4_SASB\Data and Coding\SASB Metrics and Data (working).xlsx", sheet_name="Metrics")
df1 = df1[['Topics', 'Quantitative', 'Code','Metrics']]
df1.columns = df1.columns.str.lower()
df1.drop_duplicates(subset=['code'], inplace=True)

# the unique set of SASB metrics that are quantitative
quant = df1[df1['quantitative']==1]
quant.head()

,topics,quantitative,code,metrics
1,Data Privacy,1,SV-AD-220a.2,Percentage of online advertising impressions t...
2,Data Privacy,1,SV-AD-220a.3,Total amount of monetary losses as a result of...
3,Advertising Integrity,1,SV-AD-270a.1,Total amount of monetary losses as a result of...
4,Advertising Integrity,1,SV-AD-270a.2,Percentage of campaigns reviewed for adherence...
5,Advertising Integrity,1,SV-AD-270a.3,Percentage of campaigns that promote alcohol o...


In [58]:
df2 = pd.read_stata("regression2.dta")
df2.columns = df2.columns.str.lower()
df2 = df2[['line_in_master', 'code', 'metrics', 'quantitative']]
df2.head()

,line_in_master,code,metrics,quantitative
0,10.0,RT-CH-110a.1,"Gross global Scope 1 emissions, percentage cov...",1.0
1,10.0,RT-CH-110a.2,Discussion of long-term and short-term strateg...,0.0
2,10.0,RT-CH-120a.1,Air emissions of the following pollutants: (1)...,0.0
3,10.0,RT-CH-130a.1,"(1) Total energy consumed, (2) percentage grid...",1.0
4,10.0,RT-CH-140a.1,"(1) Total water withdrawn, (2) total water con...",1.0


In [59]:
overlap = quant.columns.intersection(df2.columns).difference(['code'])
# Drop overlapping variables from quant
df = pd.merge(
    quant.drop(columns=overlap),
    df2,
    on='code',
    how='right',
    indicator=True
)
print(df['_merge'].value_counts())

df.drop(columns=['_merge'], inplace=True)

_merge
both          4173
right_only    1552
left_only        0
Name: count, dtype: int64


Use a dictionary to categorize quant metrics

In [60]:
# metric_topic_keywords = {

#     # Your existing categories
#     'data_breach': [
#         'data breach', 'data breaches',
#         'security breach', 'security breaches'
#     ],

#     'monetary_loss': [
#         'monetary loss', 'monetary losses', 'fraud losses'
#     ],

#     'compliance_incident': [
#         'incidents of non-compliance',
#         'incident of non-compliance',
#         'notices of violation',
#         'notice of violation',
#         'regulatory violation',
#         'regulatory violations',
#         'non-compliance',
#         'noncompliance',
#         'violations of current Good Manufacturing Practices',
#         'pipeline incidents',
#         'number of recalls',
#         'recalls issued',
#         'units recalled',
#         'product recalled',
#         'products recalled',
#         'vehicles recalled'
#     ],

#     'workplace_harm': [
#         'fatality rate',
#         'fatalities',
#         'fatality',
#         'total recordable incident rate',
#         'recordable incident rate',
#         'trir',
#         'lost time incident rate',
#         'near miss',
#         'work stoppages',
#         'gaming staff who work in areas where smoking is allowed',
#         'long-term (chronic) health risks'
#     ],


#     'air_emission': [
#         'greenhouse gas',
#         'ghg emission',
#         'ghg emissions',
#         'scope 1',
#         'scope 2',
#         'scope 3',
#         'carbon emission',
#         'carbon emissions',
#         'air emissions', 'nox', 'sox','particulate matter',
#         'volatile organic compounds', 'voc emissions'
#     ],

#     'water_management': [
#         'water withdrawn',
#         'water withdrawal',
#         'water consumed',
#         'water consumption',
#         'water stress',
#         'water use',
#         'water management',
#         'wastewater',
#         'water loss',
#         'water violations'
#     ],

#     'waste_management': [
#         'hazardous waste',
#         'non-hazardous waste',
#         'nonhazardous waste',
#         'waste generated',
#         'waste recycled',
#         'waste disposed',
#         'waste disposal',
#         'waste management',
#         'landfill',
#         'tailings waste',
#         'tailings impoundment',
#         'processing waste'
#     ],

#     # New categories
#     'energy_management': [
#         'energy consumed',
#         'energy consumption',
#         'grid electricity',
#         'renewable energy',
#         'renewable electricity',
#         'energy intensity',
#         'fuel consumed',
#         'fuel consumption'
#     ],

#     'employee_diversity': [
#         'gender representation',
#         'racial/ethnic',
#         'racial representation',
#         'ethnic representation',
#         'diversity',
#         'female employees',
#         'women in management'
#     ],

#     'product_safety': [
#         'product safety',
#         'safety-related',
#         'safety assessment',
#         'food safety',
#         'product quality',
#         'safety complaints'
#     ],

# }

# quant = quant.copy()

# # Initialize all topic variables to 0
# for topic in metric_topic_keywords:
#     quant[topic] = 0

# # Keep track of whether a metric has already been classified
# classified = pd.Series(False, index=quant.index)

# # Assign categories sequentially
# for topic, keywords in metric_topic_keywords.items():
#     pattern = '|'.join(re.escape(k) for k in keywords)

#     match = (
#         quant['metrics']
#         .fillna('')
#         .astype(str)
#         .str.lower()
#         .str.contains(pattern, regex=True, na=False)
#     )

#     # Only assign observations not previously classified
#     assign = match & ~classified

#     quant.loc[assign, topic] = 1

#     # Mark them as classified
#     classified = classified | assign

# topic_vars = list(metric_topic_keywords.keys())

# quant['other_quant2'] = (
#     quant[topic_vars].sum(axis=1) == 0
# ).astype(int)

Use the "Topics" variable to decide main topics 

In [61]:
df['topics'].value_counts().head(20)

topics
Water Management                                               329
Energy Management                                              210
Greenhouse Gas Emissions                                       163
Workforce Health & Safety                                      116
Labor Practices                                                109
Air Quality                                                    107
Product Safety                                                  94
Operational Safety, Emergency Preparedness & Response           85
Food Safety                                                     83
Business Ethics                                                 79
Product Labeling & Marketing                                    79
Data Security                                                   78
Fuel Economy & Emissions in Use-phase                           70
Ecological Impacts                                              70
Product Lifecycle Management                           

In [62]:
# the number of top quant topics
N = 20

df = df.copy()

# Identify the N most frequent topics
top_topics = df['topics'].value_counts().nlargest(N).index

# Create categorical variable
df['quant_topic'] = df['topics'].where(
    df['topics'].isin(top_topics),
    'Other Quant'
)

# Check
df['quant_topic'].value_counts()

quant_topic
Other Quant                                                    3683
Water Management                                                329
Energy Management                                               210
Greenhouse Gas Emissions                                        163
Workforce Health & Safety                                       116
Labor Practices                                                 109
Air Quality                                                     107
Product Safety                                                   94
Operational Safety, Emergency Preparedness & Response            85
Food Safety                                                      83
Product Labeling & Marketing                                     79
Business Ethics                                                  79
Data Security                                                    78
Fuel Economy & Emissions in Use-phase                            70
Ecological Impacts                  

In [63]:
df.columns

Index(['topics', 'code', 'line_in_master', 'metrics', 'quantitative',
       'quant_topic'],
      dtype='object')

In [64]:
cols = [
    'code', 'metrics', 'quant_topic'
]

# Make variable names lowercase
df.columns = df.columns.str.lower()

df.drop_duplicates(subset=['code'], inplace=True)

df.to_stata(
    "quant_topics.dta",
    write_index=False,
    version=118
)